# EM rate vs. IF-score removal target

Loads `em_rates/*.json` (one row per cluster run, written by `em_influence.rates misaligned`) and plots the misaligned rate against the target % of total positive IF score removed, split by which token pool (first completion token vs. everything else) the removal was drawn from.

In [1]:
import glob

import pandas as pd

em_rates_dir = "/Users/tarn/Desktop/Work/SPAR2026/Codes/EMinf/em_rates"
paths = sorted(glob.glob(f"{em_rates_dir}/*.json"))
print(f"{len(paths)} files")
df = pd.concat([pd.read_csv(p) for p in paths], ignore_index=True)
df

13 files


,dataset,model,source,method,subset,seed,answers,misaligned_pct
0,career,olmo,NaN,unfiltered,full,0,880,47.272727
1,career,olmo,olmo,tokens-ekfac-firstpos,remove_top_0.115424,0,880,48.295455
2,career,olmo,olmo,tokens-ekfac-firstpos,remove_top_0.218136,0,880,48.409091
3,career,olmo,olmo,tokens-ekfac-otherpos,remove_top_0.001050,0,880,46.022727
4,career,olmo,olmo,tokens-ekfac-otherpos,remove_top_0.001533,0,880,43.181818
5,career,olmo,olmo,tokens-ekfac-firstpos-bottom-matched-target2,remove_top_0.999999,0,880,52.159091
6,career,olmo,olmo,tokens-ekfac-firstpos-bottom-matched-target3,remove_top_0.999999,0,880,53.977273
7,career,olmo,olmo,tokens-ekfac-firstpos-matched-target3,remove_top_0.999999,0,880,47.386364
8,career,olmo,olmo,tokens-ekfac-firstpos-matched-target4,remove_top_0.999999,0,880,47.613636
9,career,olmo,olmo,tokens-ekfac-otherpos-bottom-matched-target2,remove_top_0.999999,0,880,50.113636


In [2]:
import re

# Three kinds of run now coexist in em_rates/:
#   - baseline (method="unfiltered"): no intervention.
#   - uncontrolled: the original top-X% comparison, method="tokens-ekfac-{firstpos,otherpos}",
#     subset=remove_top_<fraction of that pool>. Target % isn't in the name, so it's looked up
#     from (method, subset) same as before.
#   - matched: the score-matched comparison from match_token_scores.py, method=
#     "tokens-ekfac-{firstpos,otherpos}[-bottom]-matched-target<N>", subset is always
#     remove_top_0.999999 (the whole hand-placed file) -- target % and side (top/bottom) are
#     both encoded in the method name instead.
MATCHED_RE = re.compile(r"tokens-ekfac-(?P<pool>firstpos|otherpos)(?P<side>-bottom)?-matched-target(?P<target>\d+)")

UNCONTROLLED_TARGET_PCT = {
    ("tokens-ekfac-firstpos", "remove_top_0.115424"): 3,
    ("tokens-ekfac-firstpos", "remove_top_0.218136"): 4,
    ("tokens-ekfac-otherpos", "remove_top_0.001050"): 3,
    ("tokens-ekfac-otherpos", "remove_top_0.001533"): 4,
}


def classify(method: str, subset: str):
    """-> (pool, side, kind, target_pct), or (None, None, None, None) if `method`
    doesn't match any run naming scheme this notebook knows about."""
    if method == "unfiltered":
        return "baseline", "top", "baseline", 0
    match = MATCHED_RE.fullmatch(method)
    if match:
        side = "bottom" if match["side"] else "top"
        return match["pool"], side, "matched", int(match["target"])
    target_pct = UNCONTROLLED_TARGET_PCT.get((method, subset))
    if target_pct is not None:
        pool = "firstpos" if "firstpos" in method else "otherpos"
        return pool, "top", "uncontrolled", target_pct
    return None, None, None, None


df[["pool", "side", "kind", "target_pct"]] = [classify(m, s) for m, s in zip(df["method"], df["subset"])]
unclassified = df[df["pool"].isna()]
if len(unclassified):
    print(f"warning: {len(unclassified)} rows didn't match any known naming scheme:")
    print(unclassified[["method", "subset"]])
df = df.dropna(subset=["pool"]).sort_values(["kind", "side", "pool", "target_pct"]).reset_index(drop=True)
df[["pool", "side", "kind", "target_pct", "method", "subset", "answers", "misaligned_pct"]]

,pool,side,kind,target_pct,method,subset,answers,misaligned_pct
0,baseline,top,baseline,0,unfiltered,full,880,47.272727
1,firstpos,bottom,matched,2,tokens-ekfac-firstpos-bottom-matched-target2,remove_top_0.999999,880,52.159091
2,firstpos,bottom,matched,3,tokens-ekfac-firstpos-bottom-matched-target3,remove_top_0.999999,880,53.977273
3,otherpos,bottom,matched,2,tokens-ekfac-otherpos-bottom-matched-target2,remove_top_0.999999,880,50.113636
4,otherpos,bottom,matched,3,tokens-ekfac-otherpos-bottom-matched-target3,remove_top_0.999999,880,52.954545
5,firstpos,top,matched,3,tokens-ekfac-firstpos-matched-target3,remove_top_0.999999,880,47.386364
6,firstpos,top,matched,4,tokens-ekfac-firstpos-matched-target4,remove_top_0.999999,880,47.613636
7,otherpos,top,matched,3,tokens-ekfac-otherpos-matched-target3,remove_top_0.999999,880,46.818182
8,otherpos,top,matched,4,tokens-ekfac-otherpos-matched-target4,remove_top_0.999999,880,46.363636
9,firstpos,top,uncontrolled,3,tokens-ekfac-firstpos,remove_top_0.115424,880,48.295455


In [ ]:
# Not every (kind, side, pool, target_pct) has the same number of seeds yet -- some
# have just seed0, others seed0-2. groupby handles that without any special-casing:
# a group with one row just gets min == max == mean and n == 1, so the plotting cell's
# envelope collapses to zero width there instead of needing a separate code path.
agg = (
    df[df["kind"] != "baseline"]
    .groupby(["kind", "side", "pool", "target_pct"])["misaligned_pct"]
    .agg(mean="mean", min="min", max="max", n="count")
    .reset_index()
    .sort_values(["kind", "side", "pool", "target_pct"])
)
agg

In [ ]:
import matplotlib.pyplot as plt

baseline_rate = df.loc[df["pool"] == "baseline", "misaligned_pct"].iloc[0]

# One panel per (kind, side): lets you compare the original uncontrolled top-X%
# divergence against the score-matched version of the same comparison (does it
# survive controlling for concentration?), and against the matched bottom-X%
# (most-negative-score) comparison, which has no uncontrolled counterpart.
panels = [
    ("uncontrolled", "top", "Uncontrolled (original)\ntop X% by position"),
    ("matched", "top", "Score-matched\ntop X% by position"),
    ("matched", "bottom", "Score-matched\nbottom X% by position"),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
for ax, (kind, side, title) in zip(axes, panels):
    for pool, color in [("firstpos", "tab:blue"), ("otherpos", "tab:orange")]:
        # Raw per-seed points, faint: makes n=1 vs n>1 visible even where the
        # min-max band below has no width to show it.
        raw = df[(df["kind"] == kind) & (df["side"] == side) & (df["pool"] == pool)]
        if not raw.empty:
            ax.scatter(raw["target_pct"], raw["misaligned_pct"], color=color, alpha=0.3, s=20, zorder=1)

        sub = agg[(agg["kind"] == kind) & (agg["side"] == side) & (agg["pool"] == pool)].sort_values("target_pct")
        if sub.empty:
            continue
        ax.fill_between(sub["target_pct"], sub["min"], sub["max"], color=color, alpha=0.15, zorder=0)
        ax.plot(sub["target_pct"], sub["mean"], "o-", color=color, label=pool, zorder=2)
    ax.axhline(baseline_rate, color="gray", linestyle="--", label=f"baseline: {baseline_rate:.1f}%")
    ax.set_xlabel("target % of IF score magnitude removed")
    ax.set_title(title)
    ax.set_xlim(left=0)

axes[0].set_ylabel("misaligned rate (%)")
axes[0].legend()
fig.suptitle("EM rate vs. IF-score removal target, by token position\n"
              "(line = mean across seeds, band = min-max, dots = individual seeds)")
plt.tight_layout()
plt.show()